In [0]:
# Databricks notebook source
# =============================================================================
# MERCHANT SYNDICATE ANOMALY MART
# =============================================================================
# this mart scores every merchant on how suspicious their transaction
# patterns look. it combines category risk, customer concentration,
# velocity spikes, director linkages, and the risk profile of customers
# who shop there.
#
# grain: one row per merchant (2,200 rows)
# source: gold_base_enriched_transactions (merchant transactions only)
# joins:  gold_customer_risk_profile_cube (for customer risk scores)
# =============================================================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime

# ---------------------------------------------------------------------------
# where everything lives
# ---------------------------------------------------------------------------
catalog = "ubuntu_bank_fraud"
schema_name = "gold"

base_tbl   = f"{catalog}.{schema_name}.gold_base_enriched_transactions"
risk_tbl   = f"{catalog}.{schema_name}.gold_customer_risk_profile_cube"
target_tbl = f"{catalog}.{schema_name}.gold_merchant_syndicate_anomaly_mart"

batch_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"source (transactions): {base_tbl}")
print(f"source (customer risk): {risk_tbl}")
print(f"target:                 {target_tbl}")
print(f"batch:                  {batch_id}")

# ---------------------------------------------------------------------------
# load both source tables
# ---------------------------------------------------------------------------
base = spark.table(base_tbl)
risk = spark.table(risk_tbl)

# we only want transactions that have a merchant — everything else is noise
# for this mart. POS purchases and e-commerce are the merchant txns.
txn = base.filter(col("merchant_id").isNotNull())

merchant_txn_count = txn.count()
unique_merchants = txn.select("merchant_id").distinct().count()

print(f"\nmerchant transactions: {merchant_txn_count:,}")
print(f"unique merchants:      {unique_merchants:,}")

# ---------------------------------------------------------------------------
# make sure every column we plan to use actually exists in the source
# if any are missing, we stop immediately rather than failing later
# ---------------------------------------------------------------------------
required_columns = [
    # merchant identity
    "merchant_id", "merchant_name", "mcc", "merchant_category",
    "merch_risk_band", "mcc_risk_weight", "merch_director",
    "merch_director_id",
    # transaction attributes
    "customer_id", "transaction_id", "transaction_date",
    "transaction_amount", "hour_bucket", "is_weekend", "device_id"
]

missing = [c for c in required_columns if c not in txn.columns]
if missing:
    print(f"\n*** missing columns in source: {missing}")
    raise ValueError("cannot build merchant mart — required columns missing")

print(f"all {len(required_columns)} required columns present in source\n")

source (transactions): ubuntu_bank_fraud.gold.gold_base_enriched_transactions
source (customer risk): ubuntu_bank_fraud.gold.gold_customer_risk_profile_cube
target:                 ubuntu_bank_fraud.gold.gold_merchant_syndicate_anomaly_mart
batch:                  20260623_161717

merchant transactions: 1,209,180
unique merchants:      1,986
all 15 required columns present in source



In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# STEP 1: merchant-level aggregations
# roll up every transaction so each merchant gets one row with all their
# activity summarised — total volume, average ticket, customer count, etc.
# ---------------------------------------------------------------------------

# first calculate daily volumes per merchant — we need this for velocity
# spike detection later (peak day vs average day)
daily_stats = txn.groupBy("merchant_id", "transaction_date").agg(
    count("transaction_id").alias("daily_txn_count"),
    sum("transaction_amount").alias("daily_amount")
)

# per merchant: what is a typical day and what is the worst day
merchant_velocity = daily_stats.groupBy("merchant_id").agg(
    round(avg("daily_txn_count"), 1).alias("avg_daily_txns"),
    max("daily_txn_count").alias("peak_daily_txns"),
    round(avg("daily_amount"), 1).alias("avg_daily_amount"),
    max("daily_amount").alias("peak_daily_amount")
)

# main merchant aggregation — one row per merchant
merchants = txn.groupBy("merchant_id").agg(

    # ---- identity (take the first non-null value for each merchant) ----
    first("merchant_name", ignorenulls=True).alias("merchant_name"),
    first("mcc", ignorenulls=True).alias("mcc"),
    first("merchant_category", ignorenulls=True).alias("category"),
    first("merch_risk_band", ignorenulls=True).alias("operational_risk_band"),
    first("mcc_risk_weight", ignorenulls=True).alias("category_risk_weight"),
    first("merch_director", ignorenulls=True).alias("director_name"),
    first("merch_director_id", ignorenulls=True).alias("director_id"),

    # ---- volume metrics ----
    count("transaction_id").alias("total_transactions"),
    sum("transaction_amount").alias("total_amount"),
    round(avg("transaction_amount"), 2).alias("avg_transaction_amount"),
    min("transaction_amount").alias("smallest_transaction"),
    max("transaction_amount").alias("largest_transaction"),
    round(stddev("transaction_amount"), 2).alias("amount_stddev"),

    # ---- customer metrics ----
    countDistinct("customer_id").alias("distinct_customers"),

    # ---- device metrics ----
    countDistinct("device_id").alias("distinct_devices"),

    # ---- time patterns ----
    count(when(col("hour_bucket") == "Night", lit(1))).alias("night_transactions"),
    count(when(col("is_weekend") == 1, lit(1))).alias("weekend_transactions"),

    # ---- activity span ----
    countDistinct("transaction_date").alias("active_days"),
    min("transaction_date").alias("first_seen"),
    max("transaction_date").alias("last_seen")
)

# ---- derived ratios (calculated after the groupBy) ----
merchants = merchants \
    .withColumn("night_pct",
                when(col("total_transactions") > 0,
                     round(col("night_transactions") / col("total_transactions"), 3))
                .otherwise(0)) \
    .withColumn("weekend_pct",
                when(col("total_transactions") > 0,
                     round(col("weekend_transactions") / col("total_transactions"), 3))
                .otherwise(0)) \
    .withColumn("avg_txn_per_customer",
                when(col("distinct_customers") > 0,
                     round(col("total_transactions") / col("distinct_customers"), 1))
                .otherwise(0)) \
    .withColumn("avg_amount_per_customer",
                when(col("distinct_customers") > 0,
                     round(col("total_amount") / col("distinct_customers"), 1))
                .otherwise(0))

# ---- join velocity metrics onto the merchant aggregation ----
merchants = merchants.join(merchant_velocity, "merchant_id", "left")

# velocity spike ratio: how many times worse is the busiest day vs normal
# a ratio of 5.0 means the peak day had 5x the average daily volume
merchants = merchants.withColumn(
    "velocity_spike_ratio",
    when(col("avg_daily_txns") > 0,
         round(col("peak_daily_txns") / col("avg_daily_txns"), 1))
    .otherwise(0)
)

print(f"merchants after aggregation: {merchants.count():,}")
merchants.select("merchant_id", "merchant_name", "total_transactions", "distinct_customers").show(5, truncate=False)

merchants after aggregation: 1,986
+------------+---------------------+------------------+------------------+
|merchant_id |merchant_name        |total_transactions|distinct_customers|
+------------+---------------------+------------------+------------------+
|MERC00000212|SA WHOLESALE TRADERS |653               |645               |
|MERC00000346|SPAR MARKET          |604               |596               |
|MERC00001054|THE WATER            |590               |584               |
|MERC00001265|THE MEDICAL (PTY) LTD|617               |607               |
|MERC00001495|SA WATER STORE       |657               |652               |
+------------+---------------------+------------------+------------------+
only showing top 5 rows


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# STEP 2: customer concentration
# a merchant where 3 customers do 60% of the volume is suspicious.
# normal merchants have diverse customer bases — the top customers
# should account for only a small fraction of total volume.
# ---------------------------------------------------------------------------

# rank each customer by how much they spent at each merchant
cust_spend = txn.groupBy("merchant_id", "customer_id").agg(
    sum("transaction_amount").alias("cust_total_spent"),
    count("transaction_id").alias("cust_txn_count")
)

# number the customers within each merchant from highest spend to lowest
cust_window = Window.partitionBy("merchant_id").orderBy(col("cust_total_spent").desc())
cust_ranked = cust_spend.withColumn("spend_rank", row_number().over(cust_window))

# isolate the top 3 spenders per merchant and add up their contribution
top3 = cust_ranked.filter(col("spend_rank") <= 3).groupBy("merchant_id").agg(
    sum("cust_total_spent").alias("top3_total_spent"),
    sum("cust_txn_count").alias("top3_total_txns"),
    count("customer_id").alias("top3_customer_count")
)

# join back to the merchant table
merchants = merchants.join(top3, "merchant_id", "left") \
    .fillna(0, ["top3_total_spent", "top3_total_txns", "top3_customer_count"])

# what percentage of all volume came from the top 3 customers
merchants = merchants.withColumn(
    "customer_concentration_pct",
    when(col("total_amount") > 0,
         round((col("top3_total_spent") / col("total_amount")) * 100, 1))
    .otherwise(0)
)

print(f"merchants with concentration > 50%: {merchants.filter(col('customer_concentration_pct') > 50).count():,}")

merchants with concentration > 50%: 0


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# STEP 3: customer risk exposure
# what kind of customers shop at this merchant?
# if a merchant attracts lots of HIGH and CRITICAL risk customers,
# that is a strong signal something is wrong.
# ---------------------------------------------------------------------------

# get the unique customer-merchant pairs and attach each customer's risk score
cust_risk_at_merchant = txn \
    .select("merchant_id", "customer_id").distinct() \
    .join(
        risk.select("customer_id", "enterprise_risk_score", "enterprise_risk_band"),
        "customer_id",
        "left"
    )

# summarise the customer risk profile per merchant
merchant_cust_risk = cust_risk_at_merchant.groupBy("merchant_id").agg(
    round(avg("enterprise_risk_score"), 1).alias("avg_customer_risk_score"),
    max("enterprise_risk_score").alias("max_customer_risk_score"),
    # how many high risk customers shop here
    count(when(col("enterprise_risk_band") == "HIGH", lit(1))).alias("high_risk_customer_count"),
    # how many critical risk customers shop here
    count(when(col("enterprise_risk_band") == "CRITICAL", lit(1))).alias("critical_customer_count")
)

# join to merchants
merchants = merchants.join(merchant_cust_risk, "merchant_id", "left") \
    .fillna(0, ["high_risk_customer_count", "critical_customer_count"])

print(f"merchants with critical customers: {merchants.filter(col('critical_customer_count') > 0).count():,}")

merchants with critical customers: 1,986


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# STEP 4: director linkage — the syndicate network signal
# the source data has director names and ID numbers for each merchant.
# if the same director controls multiple merchants, those merchants
# are linked. a director with 5+ merchants is a major syndicate indicator.
# ---------------------------------------------------------------------------

# count how many merchants each director ID is associated with
director_stats = merchants \
    .filter(col("director_id").isNotNull()) \
    .groupBy("director_id") \
    .agg(
        count("merchant_id").alias("merchants_under_director"),
        collect_list("merchant_name").alias("linked_merchant_names")
    )

# attach the director network size back to each merchant
merchants = merchants \
    .join(director_stats.select("director_id", "merchants_under_director"),
          "director_id", "left") \
    .fillna(1, ["merchants_under_director"])

# a flag for investigators: this merchant shares a director with others
merchants = merchants.withColumn(
    "director_linkage_flag",
    when(col("merchants_under_director") > 1, 1).otherwise(0)
)

linked_count = merchants.filter(col("director_linkage_flag") == 1).count()
print(f"merchants with shared directors: {linked_count:,}")

merchants with shared directors: 287


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# STEP 5: merchant risk score (0 to 100)
# ---------------------------------------------------------------------------
# six components, each scaled so that only deviation above the population
# median contributes points. a perfectly average merchant scores near zero.
#
# components:
#   1. category risk (0-20)  — inherent risk of the merchant's MCC
#   2. concentration (0-20)  — few customers doing most of the volume
#   3. customer risk  (0-20)  — risky customers shopping here
#   4. velocity spike (0-15)  — peak day far above the average day
#   5. director link  (0-15)  — director controls multiple merchants
#   6. time patterns  (0-10)  — unusual night or weekend activity
# ---------------------------------------------------------------------------

# ---- find the median for each metric — this is our "normal" baseline ----
# anything at or below the median contributes zero points to that component
median_stats = merchants.approxQuantile(
    ["customer_concentration_pct", "avg_customer_risk_score",
     "velocity_spike_ratio", "night_pct", "weekend_pct"],
    [0.5], 0.01
)

med_conc      = median_stats[0][0]   # median customer concentration %
med_cust_risk = median_stats[1][0]   # median avg customer risk score
med_spike     = median_stats[2][0]   # median velocity spike ratio
med_night     = median_stats[3][0]   # median night %
med_weekend   = median_stats[4][0]   # median weekend %

# ---- find the maximums — used to scale deviation above the median ----
max_vals = merchants.agg(
    max("customer_concentration_pct").alias("max_conc"),
    max("avg_customer_risk_score").alias("max_cust_risk"),
    max("velocity_spike_ratio").alias("max_spike"),
    max("merchants_under_director").alias("max_director"),
    max("night_pct").alias("max_night"),
    max("weekend_pct").alias("max_weekend")
).collect()[0]

print("scaling reference points:")
print(f"  concentration  — median: {med_conc:.1f}  max: {max_vals['max_conc']:.1f}")
print(f"  customer risk  — median: {med_cust_risk:.1f}  max: {max_vals['max_cust_risk']:.1f}")
print(f"  velocity spike — median: {med_spike:.1f}  max: {max_vals['max_spike']:.1f}")
print(f"  night pct      — median: {med_night:.3f}  max: {max_vals['max_night']:.3f}")
print(f"  weekend pct    — median: {med_weekend:.3f}  max: {max_vals['max_weekend']:.3f}")

# ---- component 1: category risk (0 to 20 points) ----
# this is the only component that does not use zero-basing.
# mcc risk weight is a known industry standard — gaming = risky, grocery = safe.
# we want this to contribute even for average merchants because category
# risk is a legitimate standalone signal.
merchants = merchants.withColumn(
    "scr_category",
    round(col("category_risk_weight") * 20, 1)
)

# ---- component 2: customer concentration (0 to 20 points) ----
# only contributes if concentration is above the population median
merchants = merchants.withColumn(
    "scr_concentration",
    when(
        (col("customer_concentration_pct") > med_conc) & 
        (lit(max_vals["max_conc"]) > med_conc),
        round(((col("customer_concentration_pct") - med_conc) /
               (lit(max_vals["max_conc"]) - med_conc)) * 20, 1)
    ).otherwise(0)
)

# ---- component 3: customer risk exposure (0 to 20 points) ----
merchants = merchants.withColumn(
    "scr_customer_risk",
    when(
        (col("avg_customer_risk_score") > med_cust_risk) &
        (lit(max_vals["max_cust_risk"]) > med_cust_risk),
        round(((col("avg_customer_risk_score") - med_cust_risk) /
               (lit(max_vals["max_cust_risk"]) - med_cust_risk)) * 20, 1)
    ).otherwise(0)
)

# ---- component 4: velocity spike (0 to 15 points) ----
merchants = merchants.withColumn(
    "scr_velocity",
    when(
        (col("velocity_spike_ratio") > med_spike) &
        (lit(max_vals["max_spike"]) > med_spike),
        round(((col("velocity_spike_ratio") - med_spike) /
               (lit(max_vals["max_spike"]) - med_spike)) * 15, 1)
    ).otherwise(0)
)

# ---- component 5: director linkage (0 to 15 points) ----
# a director with 2 merchants gets some points, 5+ gets many more
merchants = merchants.withColumn(
    "scr_director",
    when(
        col("merchants_under_director") > 1,
        round(((col("merchants_under_director") - 1) /
               (lit(max_vals["max_director"]) - 1)) * 15, 1)
    ).otherwise(0)
)

# ---- component 6: time patterns (0 to 10 points) ----
# split between night and weekend activity
merchants = merchants \
    .withColumn("scr_night",
                when((col("night_pct") > med_night) & (lit(max_vals["max_night"]) > med_night),
                     round(((col("night_pct") - med_night) /
                            (lit(max_vals["max_night"]) - med_night)) * 5, 1))
                .otherwise(0)) \
    .withColumn("scr_weekend",
                when((col("weekend_pct") > med_weekend) & (lit(max_vals["max_weekend"]) > med_weekend),
                     round(((col("weekend_pct") - med_weekend) /
                            (lit(max_vals["max_weekend"]) - med_weekend)) * 5, 1))
                .otherwise(0)) \
    .withColumn("scr_time",
                col("scr_night") + col("scr_weekend"))

# ---- combine all components into the final score ----
merchants = merchants \
    .withColumn("merchant_risk_score_raw",
                col("scr_category") +
                col("scr_concentration") +
                col("scr_customer_risk") +
                col("scr_velocity") +
                col("scr_director") +
                col("scr_time")) \
    .withColumn("merchant_risk_score",
                round(least(col("merchant_risk_score_raw"), lit(100)), 0).cast("int"))

# ---- percentile-based risk bands ----
# instead of fixed thresholds, we use the actual score distribution
# to assign bands. this guarantees realistic proportions regardless
# of the underlying score range.
band_cutoffs = merchants.approxQuantile("merchant_risk_score",
                                         [0.65, 0.85, 0.96], 0.01)

low_cutoff    = band_cutoffs[0]   # bottom 65%  = LOW
medium_cutoff = band_cutoffs[1]   # 65th-85th   = MEDIUM
high_cutoff   = band_cutoffs[2]   # 85th-96th   = HIGH
                                   # top 4%       = CRITICAL

print(f"\npercentile-based band thresholds:")
print(f"  LOW       →  0 to {low_cutoff:.0f}   (bottom 65%)")
print(f"  MEDIUM    → {low_cutoff:.0f} to {medium_cutoff:.0f}   (65th-85th)")
print(f"  HIGH      → {medium_cutoff:.0f} to {high_cutoff:.0f}   (85th-96th)")
print(f"  CRITICAL  → {high_cutoff:.0f}+        (top 4%)")

merchants = merchants.withColumn(
    "merchant_risk_band",
    when(col("merchant_risk_score") >= high_cutoff, "CRITICAL")
    .when(col("merchant_risk_score") >= medium_cutoff, "HIGH")
    .when(col("merchant_risk_score") >= low_cutoff, "MEDIUM")
    .otherwise("LOW")
)

# ---- quick distribution check ----
print("\nrisk band distribution:")
merchants.groupBy("merchant_risk_band").count().orderBy(col("count").desc()).show()

print("score statistics:")
merchants.select("merchant_risk_score").describe().show()

scaling reference points:
  concentration  — median: 1.5  max: 2.3
  customer risk  — median: 19.3  max: 20.6
  velocity spike — median: 3.3  max: 5.7
  night pct      — median: 1.000  max: 1.000
  weekend pct    — median: 0.284  max: 0.354

percentile-based band thresholds:
  LOW       →  0 to 14   (bottom 65%)
  MEDIUM    → 14 to 20   (65th-85th)
  HIGH      → 20 to 27   (85th-96th)
  CRITICAL  → 27+        (top 4%)

risk band distribution:
+------------------+-----+
|merchant_risk_band|count|
+------------------+-----+
|               LOW| 1232|
|            MEDIUM|  390|
|              HIGH|  258|
|          CRITICAL|  106|
+------------------+-----+

score statistics:
+-------+-------------------+
|summary|merchant_risk_score|
+-------+-------------------+
|  count|               1986|
|   mean| 12.622356495468278|
| stddev| 7.7292589890427426|
|    min|                  2|
|    max|                 53|
+-------+-------------------+



In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# STEP 6: final column selection and write to gold
# only keep columns that will be useful in power bi or investigations
# ---------------------------------------------------------------------------

target = f"{catalog}.{schema_name}.gold_merchant_syndicate_anomaly_mart"

final = merchants.select(

    # ---- identity ----
    "merchant_id",
    "merchant_name",
    "mcc",
    "category",
    "operational_risk_band",
    "category_risk_weight",
    "director_name",
    "director_id",

    # ---- volume ----
    "total_transactions",
    "total_amount",
    "avg_transaction_amount",
    "smallest_transaction",
    "largest_transaction",
    "amount_stddev",
    "active_days",
    "first_seen",
    "last_seen",

    # ---- customers ----
    "distinct_customers",
    "avg_txn_per_customer",
    "avg_amount_per_customer",
    "customer_concentration_pct",
    "top3_total_spent",
    "top3_customer_count",

    # ---- customer risk ----
    "avg_customer_risk_score",
    "max_customer_risk_score",
    "high_risk_customer_count",
    "critical_customer_count",

    # ---- velocity ----
    "avg_daily_txns",
    "peak_daily_txns",
    "avg_daily_amount",
    "peak_daily_amount",
    "velocity_spike_ratio",

    # ---- time patterns ----
    "night_pct",
    "weekend_pct",

    # ---- devices ----
    "distinct_devices",

    # ---- network ----
    "merchants_under_director",
    "director_linkage_flag",

    # ---- score components (transparency for investigators) ----
    "scr_category",
    "scr_concentration",
    "scr_customer_risk",
    "scr_velocity",
    "scr_director",
    "scr_time",

    # ---- final risk ----
    "merchant_risk_score",
    "merchant_risk_band",

    # ---- audit ----
    lit(batch_id).alias("batch_id"),
    current_timestamp().alias("built_at")
)

print(f"writing to {target}")
print(f"rows:    {final.count():,}")
print(f"columns: {len(final.columns)}")

final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target)

print("done — merchant syndicate anomaly mart written")

writing to ubuntu_bank_fraud.gold.gold_merchant_syndicate_anomaly_mart
rows:    1,986
columns: 47
done — merchant syndicate anomaly mart written


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# STEP 7: validation
# run a series of sanity checks to make sure the output makes sense
# before we hand this off for power bi or downstream marts
# ---------------------------------------------------------------------------

tbl = spark.table(target)

print("=" * 55)
print("VALIDATION: gold_merchant_syndicate_anomaly_mart")
print("=" * 55)

# ---- check 1: row count should be around 2,200 ----
total = tbl.count()
print(f"\n[1] total merchants: {total:,}  (expected ~2,200)")
print(f"    {'OK' if 2100 <= total <= 2300 else 'UNEXPECTED — should be ~2,200'}")

# ---- check 2: no duplicate merchant ids ----
dupes = tbl.groupBy("merchant_id").count().filter(col("count") > 1).count()
print(f"[2] duplicate merchants: {dupes} {'OK' if dupes == 0 else 'FAIL'}")

# ---- check 3: no null merchant ids ----
nulls = tbl.filter(col("merchant_id").isNull()).count()
print(f"[3] null merchant ids: {nulls} {'OK' if nulls == 0 else 'FAIL'}")

# ---- check 4: risk band distribution (should be a pyramid) ----
print("\n[4] risk band distribution:")
band_dist = tbl.groupBy("merchant_risk_band").count() \
    .withColumn("pct", round(col("count") / total * 100, 1)) \
    .orderBy(col("count").desc())
band_dist.show()

# ---- check 5: realism thresholds ----
low_pct = tbl.filter(col("merchant_risk_band") == "LOW").count() / total * 100
crit_pct = tbl.filter(col("merchant_risk_band") == "CRITICAL").count() / total * 100
print(f"[5] realism checks:")
print(f"    LOW:      {low_pct:.1f}%  {'OK' if low_pct > 50 else 'TOO FEW — model too aggressive'}")
print(f"    CRITICAL: {crit_pct:.1f}%  {'OK' if crit_pct < 6 else 'TOO MANY — model too aggressive'}")

# ---- check 6: score statistics ----
print("\n[6] score statistics:")
tbl.select("merchant_risk_score").describe().show()

# ---- check 7: risk by category (gaming should be higher, grocery lower) ----
print("[7] average risk score by category:")
tbl.groupBy("category").agg(
    round(avg("merchant_risk_score"), 1).alias("avg_score"),
    count("*").alias("merchants")
).orderBy(col("avg_score").desc()).show(truncate=False)

# ---- check 8: director linkage ----
linked = tbl.filter(col("director_linkage_flag") == 1).count()
print(f"[8] merchants with shared directors: {linked:,} ({linked/total*100:.1f}%)")

# ---- check 9: top 10 critical — do they look like real syndicate cases ----
print("\n[9] top 10 critical risk merchants:")
tbl.filter(col("merchant_risk_band") == "CRITICAL") \
    .select(
        "merchant_name", "category", "merchant_risk_score",
        "customer_concentration_pct", "velocity_spike_ratio",
        "merchants_under_director", "avg_customer_risk_score",
        "critical_customer_count"
    ) \
    .orderBy(col("merchant_risk_score").desc()) \
    .show(10, truncate=False)

# ---- check 10: are there merchants with zero transactions ----
zero_txn = tbl.filter(col("total_transactions") == 0).count()
print(f"\n[10] merchants with zero transactions: {zero_txn} {'OK' if zero_txn == 0 else 'CHECK'}")

print("\n" + "=" * 55)
print("validation complete")
print("=" * 55)

VALIDATION: gold_merchant_syndicate_anomaly_mart

[1] total merchants: 1,986  (expected ~2,200)
    UNEXPECTED — should be ~2,200
[2] duplicate merchants: 0 OK
[3] null merchant ids: 0 OK

[4] risk band distribution:
+------------------+-----+----+
|merchant_risk_band|count| pct|
+------------------+-----+----+
|               LOW| 1232|62.0|
|            MEDIUM|  390|19.6|
|              HIGH|  258|13.0|
|          CRITICAL|  106| 5.3|
+------------------+-----+----+

[5] realism checks:
    LOW:      62.0%  OK
    CRITICAL: 5.3%  OK

[6] score statistics:
+-------+-------------------+
|summary|merchant_risk_score|
+-------+-------------------+
|  count|               1986|
|   mean| 12.622356495468278|
| stddev| 7.7292589890427426|
|    min|                  2|
|    max|                 53|
+-------+-------------------+

[7] average risk score by category:
+----------+---------+---------+
|category  |avg_score|merchants|
+----------+---------+---------+
|Gaming    |29.9     |68      